# 07 – Daten erfassen und einen DOE planen

**Lernziele:** Messfrage, Faktoren und Zielgröße festlegen, einen vollständigen faktoriellen Versuchsplan (Design of Experiments, DOE) erstellen und Messungen nachvollziehbar protokollieren. Dauer: ca. 35 Minuten. Kein Modelltraining.

## 1. Beispiel: Aushärten eines Klebstoffs

Wie beeinflussen Temperatur und Aushärtezeit die Zugfestigkeit? Alle Zahlen sind **didaktisch gewählt, keine Prozessvorgaben**. Jede Zeile ist eine neu hergestellte und anschließend zerstörend geprüfte Probe.

| Rolle | Größe | Planung |
|---|---|---|
| Faktor A | Solltemperatur | 160 / 180 °C |
| Faktor B | Sollzeit | 10 / 20 Minuten |
| Zielgröße | Zugfestigkeit | MPa, gleiche Prüfmethode |
| Konstant halten | Material, Geometrie, Vorbereitung | gleiche Spezifikation |
| Block/Störgröße | Versuchstag | alle Kombinationen an jedem der 3 Tage |
| Metadaten | Probe, Lauf, Charge, Gerät, Zeitstempel | je Versuch speichern |

Alle vier Kombinationen werden untersucht. Ein-Faktor-nach-dem-anderen würde Wechselwirkungen leicht übersehen. Drei unabhängige Proben je Kombination ermöglichen eine erste Einschätzung der Streuung. Dreimal dieselbe Probe abzulesen sind technische Wiederholmessungen, keine drei unabhängigen Versuche.


In [ ]:
from itertools import product
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
rng = np.random.default_rng(42)


## 2. Reihenfolge randomisieren und Tage als Blöcke behandeln

Innerhalb jedes Tages wird zufällig angeordnet. So werden Aufwärmen und Drift nicht systematisch einem Faktor zugeordnet. Der Seed macht den Plan reproduzierbar. Nach Temperatur zu sortieren würde die Randomisierung aufheben. Bei schwer verstellbaren Faktoren wäre eine andere Versuchsplanung nötig.


In [ ]:
kombinationen = pd.DataFrame(product([160, 180], [10, 20]), columns=['temperatur_soll_C', 'zeit_soll_min'])
plan = pd.concat([kombinationen.sample(frac=1, random_state=42+tag).assign(tag=tag)
                  for tag in [1, 2, 3]], ignore_index=True)
plan.insert(0, 'lauf_id', np.arange(1, len(plan)+1))
plan['probe_id'] = ['P%02d' % i for i in plan['lauf_id']]
assert len(plan) == 12 and plan['probe_id'].is_unique
assert plan.groupby(['temperatur_soll_C', 'zeit_soll_min']).size().eq(3).all()
display(plan)


## 3. Messprotokoll vorbereiten

Soll- und Istwerte getrennt speichern. Fehlende Messungen bleiben fehlend, niemals als 0 eintragen. Gründe und Abweichungen dokumentieren; Rohdaten nicht überschreiben. Gerät vorab prüfen, Einheiten vereinbaren, Messung nach jedem Lauf sichern. Zeitstempel mit Zeitzone erfassen.


In [ ]:
protokoll = plan.copy()
for spalte in ['temperatur_ist_C', 'zeit_ist_min', 'zugfestigkeit_MPa']:
    protokoll[spalte] = np.nan
for spalte in ['zeitstempel', 'charge', 'geraet_id', 'bedienperson', 'bemerkung']:
    protokoll[spalte] = ''
display(protokoll.head())
# Optional nach dem Befüllen unter eigenem Namen speichern:
# protokoll.to_csv('messprotokoll_doe.csv', index=False, encoding='utf-8')


## 4. Messung simulieren und auswerten

Wir erzeugen **künstliche** Istwerte und Festigkeiten mit einer Wechselwirkung. Im echten Versuch kommen diese Spalten vom Messgerät. Der Tageffekt zeigt, weshalb jede Kombination an jedem Tag gemessen wird.


In [ ]:
messung = protokoll.copy()
messung['temperatur_ist_C'] = messung['temperatur_soll_C'] + rng.normal(0, 0.4, len(plan))
messung['zeit_ist_min'] = messung['zeit_soll_min'] + rng.normal(0, 0.1, len(plan))
a = (messung['temperatur_soll_C'] - 170) / 10
b = (messung['zeit_soll_min'] - 15) / 5
messung['zugfestigkeit_MPa'] = 30 + 4*a + 2*b + 3*a*b + 0.7*(messung['tag']-2) + rng.normal(0, 0.5, len(plan))
messung['geraet_id'] = 'SIMULATION'
messung['bemerkung'] = 'synthetische Daten, kein realer Versuch'
display(messung.groupby(['temperatur_soll_C', 'zeit_soll_min'])['zugfestigkeit_MPa'].agg(['count', 'mean', 'std']))
mittel = messung.pivot_table(index='temperatur_soll_C', columns='zeit_soll_min', values='zugfestigkeit_MPa')
mittel.plot(marker='o', ylabel='Mittlere Zugfestigkeit [MPa]', xlabel='Solltemperatur [°C]', title='Wechselwirkung zwischen Temperatur und Zeit')
plt.show()
display((mittel.loc[180] - mittel.loc[160]).rename('Temperatureffekt [MPa]'))


Nicht parallele Linien deuten hier auf eine Wechselwirkung hin: Der Temperatureffekt hängt von der Zeit ab. Einzelne Mittelwerte beweisen noch keine belastbare Prozessverbesserung. Zwei Stufen zeigen keine Krümmung; dafür wären z. B. Mittelpunktversuche nötig. Andere Chargen und unabhängige Bestätigungsversuche erweitern die Aussagekraft. Der DOE gilt zunächst nur für den untersuchten Bereich.

**Mini-Aufgabe:** Wie viele Versuche ergeben 3 Faktoren mit je 2 Stufen und 3 unabhängigen Wiederholungen? Welche Metadaten brauchst du, um neue Chargen später separat zu testen?

Weiter: `uebungen/04_uebung_datenerfassung_doe.ipynb`.
